1. Chargement et Exploration des données (EDA)

Dans une première cellule, nous allons charger le dataset AI4I 2020 et observer la fameuse distribution des pannes :

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Définition du chemin relatif depuis le dossier notebooks/
DATA_PATH = "../data/raw/ai4i2020.csv"

# 1. Chargement des données
df = pd.read_csv(DATA_PATH)

print("--- Aperçu des 5 premières lignes ---")
display(df.head())

print("\n--- Vérification des valeurs manquantes ---")
print(df.isnull().sum())

print("\n--- Distribution de la cible (Machine failure) ---")
# Affichage en pourcentage pour voir le déséquilibre des classes
print(df['Machine failure'].value_counts(normalize=True) * 100)

--- Aperçu des 5 premières lignes ---


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0



--- Vérification des valeurs manquantes ---
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

--- Distribution de la cible (Machine failure) ---
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


2. Nettoyage et Préparation des features

Dans la cellule suivante, nous allons nettoyer les données en retirant les identifiants inutiles pour un modèle de Machine Learning, encoder les variables textuelles, et isoler la variable cible.

In [4]:
# 2. Nettoyage
# UDI et Product ID sont des identifiants uniques, ils n'ont pas de pouvoir prédictif
df_cleaned = df.drop(columns=['UDI', 'Product ID'])

# Séparation des caractéristiques (X) et de la cible (y)
# On retire les colonnes de pannes spécifiques (TWF, HDF, PWF, OSF, RNF) car la cible est 'Machine failure'

colonnes_a_exclure = [
    'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'
]

X = df_cleaned.drop(columns=colonnes_a_exclure)
y = df_cleaned['Machine failure']

# Identification des colonnes catégorielles et numériques
categorical_features = ['Type']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Création du préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Application de l'encodage
X_encoded = preprocessor.fit_transform(X)

# Récupération des noms des colonnes
encoded_features = preprocessor.named_transformers_['cat'].get_feature_names_out(
    categorical_features
)

feature_names = list(encoded_features) + numeric_features

# Conversion en DataFrame
X = pd.DataFrame(
    X_encoded,
    columns=feature_names,
    index=df_cleaned.index
)

print("Colonnes préparées pour l'entraînement :")
print(X.columns.tolist())

display(X.head())

Colonnes préparées pour l'entraînement :
['Type_L', 'Type_M', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']


,Type_L,Type_M,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,0.0,1.0,298.1,308.6,1551.0,42.8,0.0
1,1.0,0.0,298.2,308.7,1408.0,46.3,3.0
2,1.0,0.0,298.1,308.5,1498.0,49.4,5.0
3,1.0,0.0,298.2,308.6,1433.0,39.5,7.0
4,1.0,0.0,298.2,308.7,1408.0,40.0,9.0


3. Séparation Train/Test et Sauvegarde (Pipeline DVC)

Enfin, nous divisons les données pour l'entraînement et le test, et nous sauvegardons la version nettoyée dans le dossier data/processed/ pour que le pôle "Pipeline de données & MLOps" puisse la versionner avec DVC.

In [3]:
# 4. Séparation Train / Test
# On utilise 'stratify=y' pour s'assurer que le faible pourcentage de pannes soit bien réparti entre train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\nDimensions du jeu d'entraînement : X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Dimensions du jeu de test : X_test={X_test.shape}, y_test={y_test.shape}")

# 5. Sauvegarde des données transformées pour le suivi DVC
# (Assure-toi que le dossier ../data/processed/ existe)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\nDonnées préparées et sauvegardées avec succès dans data/processed/ !")


Dimensions du jeu d'entraînement : X_train=(8000, 7), y_train=(8000,)
Dimensions du jeu de test : X_test=(2000, 7), y_test=(2000,)

Données préparées et sauvegardées avec succès dans data/processed/ !


In [5]:
print("Distribution de la cible - Dataset initial :")
print(y.value_counts(normalize=True) * 100)

print("\nDistribution - Train :")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribution - Test :")
print(y_test.value_counts(normalize=True) * 100)

Distribution de la cible - Dataset initial :
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64

Distribution - Train :
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Distribution - Test :
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64
